In [ ]:
# Cell 00 (E046 frozen-input candidate): submit Version 5's settings (Humming dense, CUDA graphs <= 64, max-num-seqs
# 64, util 0.945; Kaggle KV 1.55M) + one rollout per task for tasks with several test inputs (ARC_MULTI_TEST) with a
# (1 + 0.5 (n - 1)) x output budget / turn cap (ARC_MULTI_BUDGET); 80 workers, KV gate 0.85.
# E128 on the 45 multi-pair eval tasks (E046 run 3): pass@2 83.1% vs 77.5% per pair at -29% tokens per task.
# ARC_CONTEXT_LIMIT = the server's max-model-len: turns never request past it (a 3-input task overflowed in run 3).
import json, os
for _k, _v in (("ARC_WORKERS", "80"),
               ("ARC_MULTI_TEST", "1"),
               ("ARC_MULTI_BUDGET", "0.5"),
               ("ARC_CONTEXT_LIMIT", "163840"),
               ("ARC_MAX_NUM_SEQS", "64"),
               ("ARC_GPU_UTIL", "0.945"),
               ("ARC_KV_ADMIT_MAX", "0.85"),
               ("ARC_KV_ADMIT_MIN_WORKERS", "16"),
               ("ARC_AUTO_REPAIR", "1"),
               ("ARC_ADMIT_MAX_WAITING", "1"),
               ("ARC_SERVER_EXTRA_ARGS", json.dumps(["--linear-backend", "humming", "--compilation-config",
                                                     json.dumps({"cudagraph_mode": "FULL",
                                                                 "max_cudagraph_capture_size": 64})]
                                                    + ["--prefix-cache-retention-interval", "512", "--decode-context-parallel-size", "4", "--cp-kv-cache-interleave-size", "64", "--cpu-offload-gb", "1", "--cpu-offload-params", "embed_tokens", "--compilation-config", "{\"cudagraph_mode\": \"FULL\", \"max_cudagraph_capture_size\": 96}", "--kv-transfer-config", "{\"kv_connector\":\"OffloadingConnector\",\"kv_role\":\"kv_both\",\"kv_connector_extra_config\":{\"cpu_bytes_to_use\":34359738368,\"offload_prompt_only\":false,\"eviction_policy\":\"lru\"}}"]))):
    os.environ.setdefault(_k, _v)
print('PYTORCH_CUDA_ALLOC_CONF from the image:', repr(os.environ.get('PYTORCH_CUDA_ALLOC_CONF')), flush=True)
_ac = [x for x in os.environ.get('PYTORCH_CUDA_ALLOC_CONF', '').split(',') if x and 'expandable_segments' not in x]
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ','.join(_ac)
# The commit run ("Save & Run All") is a smoke test: server start + 3 tasks (2 multi-test). The competition rerun
# ignores ARC_COMMIT_MODE (cell 0 always runs "submit") and never sees the task list.
if not os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    os.environ.setdefault("ARC_COMMIT_MODE", "smoke")
    os.environ.setdefault("ARC_TASK_IDS", "25ff71a9,27a28665,00576224")
print("E046 settings:", {k: os.environ.get(k) for k in ("ARC_WORKERS", "ARC_MULTI_TEST", "ARC_MULTI_BUDGET",
                                                     "ARC_CONTEXT_LIMIT", "ARC_KV_ADMIT_MAX", "ARC_MAX_NUM_SEQS", "ARC_GPU_UTIL",
                                                     "ARC_REASONING_EFFORT", "ARC_AUTO_REPAIR", "ARC_BEST_GUESS_AT",
                                                     "ARC_BEST_GUESS_TURNS", "ARC_POST_PASS_NOTE", "ARC_ADMIT_MAX_WAITING",
                                                     "ARC_STOP_NEW_S", "ARC_FORCE_AT_S", "ARC_HARD_STOP_S", "ARC_KILL_AT_S",
                                                     "ARC_SERVER_EXTRA_ARGS", "PYTORCH_CUDA_ALLOC_CONF")}, flush=True)


In [ ]:
# Cell 0: clock, paths, fallback submission (before anything that can fail).
import glob, json, os, subprocess, sys, time
T0 = float(os.environ.get("ARC_T0") or time.time()); os.environ["ARC_T0"] = str(T0)
INPUT = os.environ.get("ARC_INPUT_ROOT", "/kaggle/input")
WORK = os.environ.get("ARC_OUT", "/kaggle/working"); os.environ["ARC_OUT"] = WORK
SCRATCH = os.environ.get("ARC_SCRATCH", "/tmp/dsv41"); os.makedirs(SCRATCH, exist_ok=True); os.makedirs(WORK, exist_ok=True)
RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
# Commit runs ("Save & Run All") only check the pipeline: "fallback" writes the fallback and stops; "smoke" runs
# ARC_TASK_LIMIT tasks with short deadlines. The competition rerun always runs "submit".
COMMIT_MODE = os.environ.get("ARC_COMMIT_MODE", "fallback")
MODE = "submit" if RERUN else os.environ.get("ARC_MODE", COMMIT_MODE)

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

# Every cache and HOME under SCRATCH, overriding the image's settings: with a model attached, /root/.cache is
# read-only on Kaggle (smoke v2 and bench v2: uv "Read-only file system" in /root/.cache/uv; setdefault kept the
# image's UV_CACHE_DIR). HOME is changed only for the subprocesses (install, solver, server), not this kernel.
_C = f"{SCRATCH}/cache"
SUB_HOME = f"{SCRATCH}/home"
os.makedirs(SUB_HOME, exist_ok=True)
for _k, _v in (("UV_CACHE_DIR", f"{_C}/uv"), ("XDG_CACHE_HOME", _C), ("VLLM_CACHE_ROOT", f"{_C}/vllm"),
               ("TRITON_CACHE_DIR", f"{_C}/triton"), ("TORCHINDUCTOR_CACHE_DIR", f"{_C}/inductor"),
               ("FLASHINFER_WORKSPACE_BASE", _C), ("TORCH_EXTENSIONS_DIR", f"{_C}/torch_ext"),
               ("HF_HOME", f"{_C}/hf"), ("CUDA_CACHE_PATH", f"{_C}/nv")):
    os.environ[_k] = _v
    os.makedirs(_v, exist_ok=True)
os.environ.pop("UV_NO_CACHE", None)
CHALLENGES = os.environ.get("ARC_CHALLENGES") or find_one(f"{INPUT}/**/arc-agi_test_challenges.json")
os.environ["ARC_CHALLENGES"] = CHALLENGES
_tasks = json.load(open(CHALLENGES))
_fb = {t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in v["test"]] for t, v in _tasks.items()}
with open(f"{WORK}/submission.json.tmp", "w") as f:
    json.dump(_fb, f)
os.replace(f"{WORK}/submission.json.tmp", f"{WORK}/submission.json")
print(f"mode {MODE}; fallback submission for {len(_tasks)} tasks written at +{time.time() - T0:.0f}s", flush=True)
# The kit dataset is one tarball (Kaggle datasets do not keep nested folders reliably): unpack and check sha256.
KIT = os.environ.get("ARC_KIT")
if not KIT:
    try:
        import hashlib, tarfile
        _tgz = find_one(f"{INPUT}/**/dsv41_kit.tgz")
        KIT = f"{SCRATCH}/kit"
        with tarfile.open(_tgz) as tf:
            tf.extractall(KIT)
        _man = json.load(open(f"{KIT}/MANIFEST.json"))
        _bad = [f for f, h in _man.items() if hashlib.sha256(open(f"{KIT}/{f}", "rb").read()).hexdigest() != h]
        assert not _bad, f"kit files differ from MANIFEST: {_bad[:5]}"
        print(f"kit unpacked from {_tgz}: {len(_man)} files, sha256 ok", flush=True)
    except Exception as exc:  # without the kit nothing else can run: keep the fallback submission
        print("kit unavailable, fallback only:", type(exc).__name__, exc, flush=True)
        MODE = "fallback"
MODEL_SRC = os.environ.get("ARC_MODEL_SRC") or (os.path.dirname(find_one(f"{INPUT}/**/e043_manifest.json") or "") or None)
print("kit", KIT, "| model", MODEL_SRC, flush=True)

In [ ]:
# Cell 0b (E046, frozen inputs): the pinned solver + harness ride in the notebook, so the inputs stay exactly
# Version 5's (no new dataset version to mount). Overwrites the kit's two files after cell 0 unpacked and checked it,
# then checks hashes, required features and settings.
# Commit run ("Save & Run All"): any failure RAISES, so the commit errors and the log shows "E046 CHECK FAILED".
# Competition rerun: the same failure restores the kit's original v3 files and runs per pair at 36 workers (a scored
# submission beats an error); the log line is printed either way. The rerun path never raises (S10): a file whose
# backup is missing or cannot be copied back keeps what is in the kit (the pinned file, or the kit's own if the
# overlay never ran), and the notebook still runs.
import base64, hashlib, io, shutil, tarfile
_MT = "..."
_MT_SHA = {"solver_dsv41.py": "77fe80a31528eb1e515cc591975fc3dc0cb3d6ee6f7c67a0341df188216e6c05", "repo/evals/run_deepseek_tools.py": "882c817f2c93dd8e9ebf65c241a694549c5b7679cbacca5d16906a28dd62eecc", "repo/evals/train_check.py": "8fe361752010bba7599b24d83c4491ee462a2cffb1d9b1d018c67b90901fc982", "repo/evals/auto_repair.py": "2292369c4ad26fea4e2b3dc5e82dad31d1180192510484a4a1021a7e89b8a33d", "repo/evals/auto_repair_names.json": "571f47eb1bd077358d2153a943d0e6710c4ec18cf5828174d86c8be1fac03e92"}
_MT_EXPECT = {"ARC_WORKERS": "80", "ARC_MULTI_TEST": "1", "ARC_MULTI_BUDGET": "0.5", "ARC_CONTEXT_LIMIT": "163840", "ARC_KV_ADMIT_MAX": "0.85", "ARC_AUTO_REPAIR": "1", "ARC_ADMIT_MAX_WAITING": "1"}
_MT_NEW = ["repo/evals/auto_repair.py", "repo/evals/auto_repair_names.json"]
_MT_NEED = {"repo/evals/run_deepseek_tools.py": ["--auto-repair"], "solver_dsv41.py": ["ARC_ADMIT_MAX_WAITING"]}
if MODE != "fallback":
    _bak, _why = {}, ""
    try:
        assert KIT, "kit not unpacked"
        for _f in _MT_SHA:
            if _f in _MT_NEW:
                continue  # new to the kit (no v3 original); the v3 code never imports it
            shutil.copy2(f"{KIT}/{_f}", f"{KIT}/{_f}.v3orig")
            _bak[_f] = f"{KIT}/{_f}.v3orig"  # S10: only once the backup exists
        with tarfile.open(fileobj=io.BytesIO(base64.b64decode(_MT))) as _tf:
            _tf.extractall(KIT)
        for _f, _h in _MT_SHA.items():
            assert hashlib.sha256(open(f"{KIT}/{_f}", "rb").read()).hexdigest() == _h, f"overlay mismatch: {_f}"
        _sv = open(f"{KIT}/solver_dsv41.py").read()
        _hv = open(f"{KIT}/repo/evals/run_deepseek_tools.py").read()
        for _need in ("MULTI_TEST", "MULTI_BUDGET", "CONTEXT_LIMIT", "KV_ADMIT_MAX", "split_multi", "restart_allowed",
                      "shutdown_socket"):
            assert _need in _sv, f"solver lacks {_need}"
        for _need in ("--multi-test", "--multi-budget", "--context-limit", "class ContextFull", "import builtins as _b",
                      "def answer_items", "def context_budget"):
            assert _need in _hv, f"harness lacks {_need}"
        for _f, _needs in _MT_NEED.items():  # features the chosen opt-in settings rely on
            _txt = open(f"{KIT}/{_f}").read()
            for _need in _needs:
                assert _need in _txt, f"{_f} lacks {_need}"
        for _k, _v in _MT_EXPECT.items():
            assert os.environ.get(_k) == _v, f"{_k}={os.environ.get(_k)!r}, expected {_v!r}"
    except Exception as exc:
        _why = f"{type(exc).__name__}: {exc}"
    if not _why:
        print("E046 CHECK OK: multi-test code installed and checked:", {f: h[:8] for f, h in _MT_SHA.items()},
              _MT_EXPECT, flush=True)
    else:
        print("=" * 100, f"E046 CHECK FAILED: {_why}", "=" * 100, sep=chr(10), flush=True)
        if not RERUN:
            raise RuntimeError(f"E046 CHECK FAILED (commit run): {_why}")
        _kept = []
        try:
            for _f in _MT_SHA:  # rerun: back to the kit's v3 files, per pair, 36 workers
                if _f in _MT_NEW:
                    continue
                try:
                    shutil.copy2(_bak[_f], f"{KIT}/{_f}.restore")
                    os.replace(f"{KIT}/{_f}.restore", f"{KIT}/{_f}")
                except Exception as exc:
                    _kept.append(f"{_f} ({type(exc).__name__})")
            # (a kept pinned solver would pass the opt-in harness flags to the v3 harness, whose parser exits)
            for _k in ("ARC_MULTI_TEST", "ARC_MULTI_BUDGET", "ARC_KV_ADMIT_MAX", "ARC_CONTEXT_LIMIT", "ARC_AUTO_REPAIR",
                       "ARC_BEST_GUESS_AT", "ARC_BEST_GUESS_TURNS", "ARC_POST_PASS_NOTE", "ARC_REASONING_EFFORT",
                       "ARC_ADMIT_MAX_WAITING"):
                os.environ.pop(_k, None)
            os.environ["ARC_WORKERS"] = "36"
            os.environ["ARC_SERVER_EXTRA_ARGS"] = "[\"--linear-backend\", \"humming\", \"--compilation-config\", \"{\\\"cudagraph_mode\\\": \\\"FULL\\\", \\\"max_cudagraph_capture_size\\\": 64}\"]"  # no flag that needs the new overlay
        except Exception as exc:
            _kept.append(f"{type(exc).__name__}: {exc}")
        print("E046 FALLBACK: the kit's v3 code, per pair, 36 workers"
              + (f"; not restored, kept as is: {', '.join(_kept)}" if _kept else ""), flush=True)


In [ ]:
# Cell 1: stage the checkpoint to local disk in the background (stage_model.py), while vLLM installs.
# "warm" (read the mount once, no writes): smoke v2 (2026-10-24) with "copy" hit "Read-only file system" on the
# root overlay during the install while 131 GB were being written to /tmp.
STAGE_MODE = os.environ.get("ARC_STAGE_MODE", "warm")
STAGE_DONE = f"{SCRATCH}/stage_done"
if MODE != "fallback" and MODEL_SRC:
  try:
    if os.path.exists(STAGE_DONE):
        os.remove(STAGE_DONE)
    _stage = subprocess.Popen([sys.executable, "-I", f"{KIT}/stage_model.py", MODEL_SRC, f"{SCRATCH}/model", STAGE_DONE,
                               STAGE_MODE, os.environ.get("ARC_STAGE_THREADS", "16")],
                              stdout=open(f"{SCRATCH}/stage.log", "w"), stderr=subprocess.STDOUT,
                              stdin=subprocess.DEVNULL, start_new_session=True)
    print(f"staging ({STAGE_MODE}) started, pid {_stage.pid}", flush=True)
  except Exception as exc:  # the solver then loads from MODEL_SRC (no staging marker)
    STAGE_DONE = ""
    print("staging failed to start:", type(exc).__name__, exc, flush=True)

In [ ]:
# Cell 2: offline install of the fork wheel set, the vLLM overlay, the CUDA 13 shim (E042's Kaggle recipe).
import hashlib, shutil, tarfile
V = os.environ.get("ARC_VENV", f"{SCRATCH}/venv"); PY = f"{V}/bin/python"; SP = f"{V}/lib/python3.12/site-packages"
BASE_ENV = dict(os.environ)
def sh(cmd, timeout=None, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, errors="replace", timeout=timeout, env=env)
    return r.returncode, (r.stdout or "") + (r.stderr or "")
if MODE != "fallback" and not os.environ.get("ARC_SKIP_INSTALL"):
  try:
    real = "/usr/local/nvidia/lib64"  # host driver libcuda first (the image also has an older compat libcuda)
    if os.path.exists(real + "/libcuda.so"):
        os.environ["LD_LIBRARY_PATH"] = real + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_PRELOAD"] = real + "/libcuda.so"
    WHEELS = os.path.dirname(find_one(f"{INPUT}/**/vllm-0.28.1rc1*.whl"))
    t = time.time()
    _uvc = os.environ["UV_CACHE_DIR"]
    _ienv = dict(os.environ, HOME=SUB_HOME, UV_CACHE_DIR=_uvc, TMPDIR=f"{SCRATCH}/tmp")
    os.makedirs(f"{SCRATCH}/tmp", exist_ok=True)
    if shutil.which("uv"):
        rc, out = sh(f"uv venv --cache-dir {_uvc} --python {sys.executable} {V} && uv pip install --cache-dir {_uvc} "
                     f"--python {PY} --no-index --no-deps --find-links {WHEELS} -r {WHEELS}/requirements.lock",
                     timeout=3600, env=_ienv)
    else:
        rc, out = sh(f"{sys.executable} -m venv {V} && {PY} -m pip install --no-cache-dir --no-index --no-deps "
                     f"--find-links {WHEELS} -r {WHEELS}/requirements.lock", timeout=3600, env=_ienv)
    print(f"offline install rc {rc} in {time.time() - t:.0f}s", out[-800:], flush=True)
    assert rc == 0, "vLLM install failed"
    with tarfile.open(f"{KIT}/vllm_overlay_rw.tgz") as tf:
        tf.extractall(SP)
    for line in open(f"{KIT}/vllm_overlay_rw.sha256"):
        h, rel = line.split()
        assert hashlib.sha256(open(f"{SP}/{rel}", "rb").read()).hexdigest() == h, f"overlay mismatch: {rel}"
    print("vLLM overlay installed and checked", flush=True)
    cu = f"{SP}/nvidia/cu13"; shim = f"{SCRATCH}/cuda13"
    os.makedirs(f"{shim}/lib64/stubs", exist_ok=True)
    for d in ("bin", "include", "nvvm", "lib", "share"):
        if os.path.exists(f"{cu}/{d}") and not os.path.lexists(f"{shim}/{d}"):
            os.symlink(f"{cu}/{d}", f"{shim}/{d}")
    for f in os.listdir(f"{cu}/lib"):
        if not os.path.lexists(f"{shim}/lib64/{f}"):
            os.symlink(f"{cu}/lib/{f}", f"{shim}/lib64/{f}")
    for name in ("cudart", "nvrtc"):
        r = sorted(glob.glob(f"{cu}/lib/lib{name}.so.*"))
        if r and not os.path.lexists(f"{shim}/lib64/lib{name}.so"):
            os.symlink(r[0], f"{shim}/lib64/lib{name}.so")
    drv = next(iter(sorted(glob.glob("/usr/local/nvidia/lib64/libcuda.so*")) + sorted(glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*"))), None)
    for dst in (f"{shim}/lib64/stubs/libcuda.so", f"{shim}/lib64/libcuda.so"):
        if drv and not os.path.lexists(dst):
            os.symlink(drv, dst)
    BASE_ENV = dict(os.environ)
    BASE_ENV.update(CUDA_HOME=shim, CUDA_PATH=shim, PATH=f"{V}/bin:{shim}/bin:" + os.environ["PATH"],
                    LD_LIBRARY_PATH=os.environ.get("LD_LIBRARY_PATH", "") + f":{shim}/lib64")
    rc, out = sh(f'nvcc --version | tail -1; {PY} -c "import torch, vllm, flashinfer, importlib.metadata as m; '
                 f'print(torch.__version__, torch.cuda.device_count(), vllm.__version__, flashinfer.__version__, '
                 f'm.version(\'humming-kernels\'))"', timeout=900, env=BASE_ENV)
    print("toolchain / imports:", rc, out[-600:], flush=True)
    rc, out = sh("free -g | head -2; df -h /tmp /kaggle/working | tail -2; nproc; cat /sys/fs/cgroup/memory.max "
                 "/sys/fs/cgroup/memory.oom.group 2>/dev/null; grep -m1 'model name' /proc/cpuinfo", timeout=60)
    print(out, flush=True)
  except Exception as exc:  # no working vLLM: keep the fallback submission
    print("install failed, fallback only:", type(exc).__name__, exc, flush=True)
    MODE = "fallback"
# The tool code runs on the image's Python (scipy etc.); if that cannot import numpy/scipy in the executor's
# environment (-I, PATH only), use the venv's Python instead (numpy only).
TOOL_PY = os.environ.get("ARC_TOOL_PYTHON", sys.executable)
if MODE != "fallback":
    try:
        r = subprocess.run([TOOL_PY, "-I", "-c", "import numpy, scipy.ndimage; print(numpy.__version__)"],
                           env={"PATH": os.environ.get("PATH", ""), "OMP_NUM_THREADS": "1"}, capture_output=True,
                           text=True, timeout=300)
        print("tool python", TOOL_PY, "->", r.returncode, (r.stdout + r.stderr).strip()[-300:], flush=True)
        if r.returncode != 0 and os.path.exists(PY):
            TOOL_PY = PY
    except Exception as exc:
        print("tool python check failed:", exc, flush=True)
BASE_ENV.pop("DSV41_SYNTH_INIT", None)  # the overlay's synthetic init must stay off for real weights
BASE_ENV["HOME"] = SUB_HOME  # /root/.cache is read-only on Kaggle with a model attached
# FlashInfer's sampling module cannot JIT on the Kaggle image (E042 K-runs); vLLM's PyTorch sampler instead.
BASE_ENV["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

In [ ]:
# Cell 3: the solver (vLLM server, tool harness, deadlines, watchdog, incremental submission).
if MODE != "fallback" and MODEL_SRC:
  try:
    import signal, threading
    env = dict(BASE_ENV)
    env.update(ARC_KIT=KIT, ARC_MODEL_DIR=MODEL_SRC, ARC_STAGE_DONE=STAGE_DONE, ARC_VLLM=f"{V}/bin/vllm",
               ARC_TOOL_PYTHON=TOOL_PY, ARC_SCRATCH=SCRATCH)
    if MODE == "smoke":
        env.setdefault("ARC_TASK_LIMIT", "8")
        for k, v in (("ARC_STOP_NEW_S", "4800"), ("ARC_FORCE_AT_S", "5400"), ("ARC_HARD_STOP_S", "5700")):
            env.setdefault(k, v)
    p = subprocess.Popen([PY, "-u", f"{KIT}/solver_dsv41.py"], env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, errors="replace", start_new_session=True)
    # Independent kill switch: whatever hangs inside the solver, its whole session dies at T0 + ARC_KILL_AT_S
    # (default 11h45; the solver's own hard stop is 11h30), so cell 4 and the notebook end before Kaggle's 12 h.
    def _kill():
        try:
            os.killpg(p.pid, signal.SIGKILL)
            print("kill switch: solver session killed", flush=True)
        except Exception:
            pass
    _timer = threading.Timer(max(0.0, T0 + float(os.environ.get("ARC_KILL_AT_S", 42300)) - time.time()), _kill)
    _timer.daemon = True
    _timer.start()
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    _timer.cancel()
    print("solver exit code", p.returncode, flush=True)
  except Exception as exc:
    print("solver cell failed:", type(exc).__name__, exc, flush=True)

In [ ]:
# Cell 4: final check of submission.json; log archive.
import tarfile
def _ok(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) and 0 < len(r) <= 30 for r in g)
            and len({len(r) for r in g}) == 1 and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))
try:
    sub = json.load(open(f"{WORK}/submission.json"))
    assert set(sub) == set(_tasks), f"task ids differ ({len(sub)} vs {len(_tasks)})"
    for t, v in _tasks.items():
        assert len(sub[t]) == len(v["test"]), f"{t}: {len(sub[t])} rows for {len(v['test'])} test inputs"
        assert all(_ok(r["attempt_1"]) and _ok(r["attempt_2"]) for r in sub[t]), f"{t}: invalid grid"
    print("submission.json valid", flush=True)
except Exception as exc:
    print("submission.json invalid, writing the fallback:", exc, flush=True)
    json.dump(_fb, open(f"{WORK}/submission.json", "w"))
try:
    logs = f"{WORK}/dsv41_logs"
    if os.path.isdir(f"{logs}/traces"):
        with tarfile.open(f"{WORK}/dsv41_traces.tar.gz", "w:gz") as tf:
            tf.add(f"{logs}/traces", arcname="traces")
        shutil.rmtree(f"{logs}/traces", ignore_errors=True)
    for f in ("stage.log",):
        if os.path.exists(f"{SCRATCH}/{f}"):
            shutil.copy(f"{SCRATCH}/{f}", f"{logs}/{f}" if os.path.isdir(logs) else f"{WORK}/{f}")
except Exception as exc:
    print("log archive failed:", exc, flush=True)
print(f"done at +{(time.time() - T0) / 3600:.2f}h", flush=True)